# MECHRABOT — Chunking Pipeline

**Input:** `saved_document.json` (DoclingDocument with REFERENCED images)

**Output:** `all_chunks.json` — ready for BGE-m3 embedding + Qdrant indexing

### Pipeline Steps:
1. Install dependencies
2. Load DoclingDocument from JSON
3. Build page → image_paths lookup
4. HybridChunker for text/list/heading/table chunks
5. Custom row-per-chunk for torque/spec tables
6. Merge all chunks with image_paths + metadata
7. Post-clean (ftfy, filter noise, unique IDs)
8. Save & verify

## Step 1 — Install Dependencies

In [ ]:
pip install --upgrade docling ftfy

## Step 2 — Load DoclingDocument

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
from docling_core.types.doc.document import DoclingDocument

# UPDATE THIS PATH to match your Kaggle dataset
JSON_PATH = "/kaggle/input/datasets/ahmedezzattaha/saved-doc-refrenced-json/saved_document.json"

doc = DoclingDocument.model_validate_json(open(JSON_PATH).read())

print(f"Loaded: {doc.name}")
print(f"Pages:    {len(doc.pages)}")
print(f"Texts:    {len(doc.texts)}")
print(f"Tables:   {len(doc.tables)}")
print(f"Pictures: {len(doc.pictures)}")

## Step 3 — Build page → image_paths lookup

This is the key step: instead of creating separate picture chunks,
we build a dictionary that maps each page number to the image files on that page.
Later, when we create text chunks, we attach the relevant images via this lookup.

In [ ]:
from collections import defaultdict

# Build lookup: page_number → list of image file paths
page_images = defaultdict(list)

for pic in doc.pictures:
    page = pic.prov[0].page_no if pic.prov else None
    uri  = str(pic.image.uri) if pic.image else None
    if page and uri:
        page_images[page].append(uri)

print(f"Pages with images: {len(page_images)}")
print(f"Total image links: {sum(len(v) for v in page_images.values())}")

# Sample: show which images are on page 5
print(f"\nExample — page 5 images: {page_images.get(5, 'none')}")

## Step 4 — HybridChunker (text, lists, headings, most tables)

Uses BAAI/bge-m3 tokenizer (your embedding model) to ensure chunks fit within 512 tokens.
- `repeat_table_header=True` → keeps column context when large tables are split
- `merge_peers=True` → merges tiny sibling chunks into coherent paragraphs
- `always_emit_headings=False` → drops heading-only empty chunks automatically

In [ ]:
from docling_core.transforms.chunker import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer

tokenizer = HuggingFaceTokenizer.from_pretrained(
    model_name="BAAI/bge-m3",
    max_tokens=512
)

chunker = HybridChunker(
    tokenizer=tokenizer,
    repeat_table_header=True,
    merge_peers=True,
    always_emit_headings=False
)

text_chunks = list(chunker.chunk(doc))
print(f"HybridChunker produced: {len(text_chunks)} chunks")

## Step 5 — Custom row-per-chunk for torque/spec tables

HybridChunker handles most tables well, but torque/clearance spec tables need
each row as its own chunk so BM25 sparse search can hit exact values like `55 Nm`.

In [ ]:
import re

spec_pattern = re.compile(r'\b(\d+\.?\d*)\s*(Nm|mm|bar|kPa|MPa)\b')
spec_chunks = []

for table in doc.tables:
    # Skip TOC/index tables
    if str(table.label) == "document_index":
        continue
    
    grid = table.data.grid
    if not grid or len(grid) < 2:
        continue
    
    # Only target tables with engineering values
    flat_text = " ".join(c.text for row in grid for c in row)
    if not spec_pattern.search(flat_text):
        continue
    
    # Get page for this table
    table_page = table.prov[0].page_no if table.prov else None
    
    # Extract headers from first row
    headers = [c.text for c in grid[0]]
    
    # Build breadcrumb from captions or fallback
    breadcrumb = " > ".join(table.captions or ["Specs"])
    
    # Create one chunk per data row
    for row in grid[1:]:
        pairs = []
        for i in range(min(len(headers), len(row))):
            if row[i].text.strip():
                pairs.append(f"{headers[i]}: {row[i].text}")
        if pairs:  # skip empty rows
            chunk_text = f"[{breadcrumb}]\n" + " | ".join(pairs)
            spec_chunks.append({
                "text": chunk_text,
                "page": table_page
            })

print(f"Spec table row chunks: {len(spec_chunks)}")

# Preview
if spec_chunks:
    print(f"\nExample: {spec_chunks[0]['text'][:120]}")

## Step 6 — Merge all chunks with image_paths + metadata

Each chunk gets:
- `content` — the cleaned text (ftfy-fixed)
- `chunk_id` — unique deterministic ID
- `source` — original PDF filename
- `page` — page number
- `section` — heading breadcrumb from HybridChunker
- `type` — "text" or "table_spec"
- `image_paths` — list of image file paths from the SAME page

In [ ]:
import ftfy
import hashlib
import json

all_chunks = []

# --- Text chunks (from HybridChunker) ---
for i, c in enumerate(text_chunks):
    # Get page number
    try:
        page = c.meta.doc_items[0].prov[0].page_no
    except (IndexError, AttributeError):
        page = None
    
    # Build chunk
    all_chunks.append({
        "content": ftfy.fix_text(c.text),
        "meta": {
            "chunk_id": hashlib.md5(f"{i}_{c.text[:80]}".encode()).hexdigest()[:12],
            "source": "m11_SM.pdf",
            "page": page,
            "section": c.meta.headings,
            "type": "text",
            "image_paths": page_images.get(page, [])  # ← images linked by page
        }
    })

# --- Spec table row chunks ---
for i, sc in enumerate(spec_chunks):
    page = sc["page"]
    all_chunks.append({
        "content": ftfy.fix_text(sc["text"]),
        "meta": {
            "chunk_id": hashlib.md5(f"spec_{i}_{sc['text'][:80]}".encode()).hexdigest()[:12],
            "source": "m11_SM.pdf",
            "page": page,
            "section": None,
            "type": "table_spec",
            "image_paths": page_images.get(page, [])  # ← spec tables get images too
        }
    })

print(f"Total chunks: {len(all_chunks)}")

## Step 7 — Post-clean: filter noise & verify quality

In [ ]:
# Remove very short/noisy chunks (image codes, lone page numbers)
before = len(all_chunks)
all_chunks = [c for c in all_chunks if len(c["content"].strip()) >= 30]
print(f"Removed {before - len(all_chunks)} noisy short chunks")
print(f"Final chunk count: {len(all_chunks)}")

# Strip leftover image artifact codes from text (BESM010007, VISM130001T etc.)
img_code = re.compile(r'\b[A-Z]{2,6}\d{5,}[A-Z]?\b')
for c in all_chunks:
    c["content"] = img_code.sub('', c["content"]).strip()
    c["content"] = re.sub(r'\n{3,}', '\n\n', c["content"])  # collapse extra newlines

In [ ]:
# --- Quality checks ---
from collections import Counter

types = Counter(c['meta']['type'] for c in all_chunks)
ids = [c['meta']['chunk_id'] for c in all_chunks]
with_imgs = sum(1 for c in all_chunks if c['meta']['image_paths'])
lengths = [len(c['content']) for c in all_chunks]

print("=== QUALITY REPORT ===")
print(f"Types:              {dict(types)}")
print(f"Unique chunk_ids:   {len(set(ids))} / {len(ids)}")
print(f"Chunks with images: {with_imgs} ({100*with_imgs/len(all_chunks):.1f}%)")
print(f"Content length:     min={min(lengths)}, max={max(lengths)}, median={sorted(lengths)[len(lengths)//2]}")
print(f"Pages covered:      {min(c['meta']['page'] for c in all_chunks if c['meta']['page'])} - {max(c['meta']['page'] for c in all_chunks if c['meta']['page'])}")

In [ ]:
# --- Preview samples ---

# Text chunk WITH images
for c in all_chunks:
    if c['meta']['type'] == 'text' and c['meta']['image_paths']:
        print("=== TEXT CHUNK WITH IMAGES ===")
        print(json.dumps(c, indent=2, ensure_ascii=False)[:600])
        break

print()

# Spec chunk
for c in all_chunks:
    if c['meta']['type'] == 'table_spec':
        print("=== SPEC TABLE CHUNK ===")
        print(json.dumps(c, indent=2, ensure_ascii=False)[:400])
        break

## Step 8 — Save output

In [ ]:
with open("all_chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"✅ Saved {len(all_chunks)} chunks → all_chunks.json")